# Session 07 answers
Try first. Relative paths start at cwd; spaces are valid. with closes handles. CSV fields initially read as strings. This reference notebook only reads; use the scripts for writing.

## Locate shared fixtures
Works from this notebook folder or course root. Expected True.

In [ ]:
from pathlib import Path
import csv
course_root = Path.cwd()
if not (course_root / "datasets").is_dir():
    course_root = Path.cwd().parents[2]
data_folder = course_root / "datasets" / "session-07"
print(data_folder.is_dir())
with open(data_folder / "sequence.txt", encoding="utf-8") as handle:
    sequence = handle.read().strip().upper()
print(sequence, len(sequence))
with open(data_folder / "sequences.csv", newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        print(row["id"], len(row["sequence"]))

## Reader and report reference
Functions validate before writing. Description words are discarded deliberately, IDs retained. Blank lines ignored; invalid data raises a specific error.

In [ ]:
"""Limited educational FASTA reader and validated CSV writer; synthetic DNA only."""
import csv

### Build validate_dna

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def validate_dna(sequence):
    """Check already-normalized nonempty A/T/G/C text."""
    if sequence == "":
        return False
    for base in sequence:
        if base not in "ATGC":
            return False
    return True

### Build read_fasta

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def read_fasta(path):
    """Read unique first-token IDs and wrapped DNA; raise ValueError on bad data."""
    records = {}
    current_id = None
    with open(path, encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line == "":
                continue
            if line.startswith(">"):
                # Reject an empty previous record before moving to the next.
                if current_id is not None and records[current_id] == "":
                    raise ValueError("Empty sequence: " + current_id)
                header = line[1:].strip()
                if header == "":
                    raise ValueError("Empty FASTA header")
                current_id = header.split()[0]
                if current_id in records:
                    raise ValueError("Duplicate identifier: " + current_id)
                records[current_id] = ""
            else:
                if current_id is None:
                    raise ValueError("Sequence before first header")
                sequence_line = line.upper()
                if not validate_dna(sequence_line):
                    raise ValueError("Invalid DNA symbols: " + current_id)
                records[current_id] = records[current_id] + sequence_line
    if len(records) == 0:
        raise ValueError("No FASTA records")
    if records[current_id] == "":
        raise ValueError("Empty sequence: " + current_id)
    return records

### Build summarize_records

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def summarize_records(records):
    """Validate all records before output; return rows with counts and GC percent."""
    if len(records) == 0:
        raise ValueError("No records to report")
    rows = []
    for sequence_id in records:
        sequence = records[sequence_id]
        if sequence_id.strip() == "" or not validate_dna(sequence):
            raise ValueError("Invalid report record: " + sequence_id)
        counts = {"A": 0, "T": 0, "G": 0, "C": 0}
        for base in sequence:
            counts[base] = counts[base] + 1
        gc_percent = (counts["G"] + counts["C"]) / len(sequence) * 100
        rows.append([sequence_id, len(sequence), counts["A"], counts["T"],
                     counts["G"], counts["C"], gc_percent])
    return rows

### Build save_report

Read its contract first. Trace the input, each decision and the returned result before moving to the next function.

In [ ]:
def save_report(records, output_path):
    """Check every record before opening the output; overwrite only the named report."""
    rows = summarize_records(records)
    with open(output_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "length", "A", "T", "G", "C", "gc_percent"])
        writer.writerows(rows)

## Homework hand checks
Seed_A 4/50%; Seed_B 4/0%; Seed_C 6/100%. Compare counts too.

In [ ]:
records = read_fasta(data_folder / "homework.fasta")
rows = summarize_records(records)
print(rows)
assert rows == [["Seed_A",4,1,1,1,1,50.0], ["Seed_B",4,2,2,0,0,0.0], ["Seed_C",6,0,0,3,3,100.0]]
for name in ["missing.fasta", "empty.txt", "invalid.fasta", "duplicate.fasta", "no_header.fasta"]:
    try:
        read_fasta(data_folder / name)
    except FileNotFoundError:
        print("Missing:", name)
    except ValueError as error:
        print("Invalid:", name, error)